# CNN de gatos y perros · Entrenamiento y predicción en Kaggle

Notebook autónomo para ejecutar la CNN con un **dataset ya procesado y dividido**.
Carga directamente las imágenes, entrena, guarda el mejor modelo, muestra las curvas,
evalúa en test y permite probar una imagen nueva al final.

Adjunta tu dataset a Kaggle, selecciona una GPU si está disponible y configura
`DATA_DIR` en la primera celda. No se preparan ni copian imágenes, no se divide el
dataset y no se realizan inspecciones del dataset ni de la arquitectura.
No se requiere `manifest.json`.

Estructura esperada (imágenes RGB de 224 × 224):

```text
split/
├── train/
│   ├── cat/
│   └── dog/
├── val/
│   ├── cat/
│   └── dog/
└── test/
    ├── cat/
    └── dog/
```

Las carpetas de cada clase contienen sus imágenes. Los resultados se guardan en
`/kaggle/working` al ejecutar en Kaggle, o en `artifacts/kaggle` al ejecutar localmente.

## 1. Bibliotecas y configuración

Cambia `DATA_DIR` por la carpeta que contiene `train`, `val` y `test`.
`SMOKE_TEST=True` prueba una época con dos lotes por conjunto; sus métricas no
representan el rendimiento final. Usa `False` para entrenar con todos los datos.

Para evaluar un modelo terminado, usa `RUN_TRAINING=False` y configura
`EXISTING_CHECKPOINT`. La predicción manual está en el último bloque.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cnn")

import csv
import json
import math
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from PIL import Image, ImageOps
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

ON_KAGGLE = Path("/kaggle/input").is_dir()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts/kaggle")
DATA_DIR = Path("/kaggle/input/mi-dataset/split") if ON_KAGGLE else Path("dataset/split")

SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4           # 0 desactiva esta regularización
DEVICE_NAME = "auto"         # auto, cpu o cuda; usa una GPU si hay CUDA
SMOKE_TEST = False
RUN_TRAINING = True          # False: omite entrenamiento y permite evaluar un checkpoint
EXISTING_CHECKPOINT = None   # Necesario para evaluar/predecir sin entrenar

RUN_DIR = WORK_ROOT / ("run_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
MAX_BATCHES = 2 if SMOKE_TEST else None
TRAIN_EPOCHS = 1 if SMOKE_TEST else EPOCHS
EFFECTIVE_BATCH_SIZE = 2 if SMOKE_TEST else BATCH_SIZE

for name, value in (("EPOCHS", EPOCHS), ("BATCH_SIZE", BATCH_SIZE)):
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} debe ser un entero positivo")
if not math.isfinite(LEARNING_RATE) or LEARNING_RATE <= 0:
    raise ValueError("LEARNING_RATE debe ser positivo y finito")
if not math.isfinite(WEIGHT_DECAY) or WEIGHT_DECAY < 0:
    raise ValueError("WEIGHT_DECAY debe ser no negativo y finito")
if DEVICE_NAME not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE_NAME debe ser auto, cpu o cuda")
print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)
print("Entorno:", "Kaggle" if ON_KAGGLE else "Local")
print("Datos preparados:", DATA_DIR)
print("Resultados:", RUN_DIR)

## 2. Utilidades de carga

Se conserva el preprocesamiento del entrenamiento: RGB 224 × 224 y tensores en [0, 1],
sin aumentación ni normalización adicional. Si las imágenes ya tienen el tamaño
correcto, no se redimensionan. Se leen al solicitar cada lote; no se abren todas
previamente para inspeccionarlas. Solo se mezclan las imágenes de train.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

def image_tensor(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    if image.size != (224, 224):
        image = image.resize((224, 224), Image.Resampling.LANCZOS)
    return ToTensor()(image)

def load_dataset(root, split):
    dataset = ImageFolder(Path(root) / split, transform=image_tensor)
    if dataset.class_to_idx != {"cat": 0, "dog": 1}:
        raise ValueError(f"Clases inesperadas en {split}: {dataset.class_to_idx}")
    return dataset

def make_loader(dataset, batch_size, *, shuffle=False, seed=42):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)

def new_output(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    if any(path.iterdir()):
        raise ValueError(f"La salida contiene resultados; elige otra con --output: {path}")
    return path

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

## 3. Modelo y dispositivo

CNN con cuatro convoluciones, BatchNorm, pooling global y Dropout. Salida de dos
clases: `cat=0` y `dog=1`. Se utiliza CUDA si está disponible; en caso contrario, CPU.
Los checkpoints de la arquitectura antigua de dos convoluciones no son compatibles.

In [ ]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Cuatro bloques convolucionales con normalización y reducción espacial.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )

        # Capa totalmente conectada para clasificación binaria
        self.classifier = nn.Sequential(
            # El pooling global deja una característica por cada uno de los 256 canales.
            nn.Flatten(), nn.Linear(256, 128), nn.ReLU(),
            # Dropout con probabilidad de 0.5 para evitar sobreajuste
            nn.Dropout(0.5), nn.Linear(128, 2),
        )

    # Método forward que define cómo los datos pasan a través del modelo
    def forward(self, images):
        return self.classifier(self.features(images))

def select_device(name="auto"):
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA no está disponible; usa --device cpu.")
    if name == "cpu":
        torch.set_num_threads(min(4, torch.get_num_threads()))
    return torch.device(name)

def load_checkpoint(path: Path, device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["class_to_idx"] != {"cat": 0, "dog": 1}:
        raise ValueError("El checkpoint no tiene las clases esperadas.")
    model = CatDogCNN()
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.to(device).eval(), checkpoint

In [ ]:
device = select_device(DEVICE_NAME)
seed_everything(SEED)
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
RUN_DIR = new_output(RUN_DIR)

## 4. Cargar el dataset dividido

`ImageFolder` carga las rutas y etiquetas directamente de las carpetas de clases.
No se comprueban conteos fijos, balance, tamaños ni duplicados. Train y val se cargan
solo para entrenar; test se carga en la evaluación. Si omites entrenamiento y solo
quieres predecir, puedes ejecutar el último bloque sin cargar ningún dataset.

In [ ]:
datasets = {}
if RUN_TRAINING:
    datasets["train"] = load_dataset(DATA_DIR, "train")
    datasets["val"] = load_dataset(DATA_DIR, "val")
    print(f"Train: {len(datasets['train'])}; validación: {len(datasets['val'])}")
else:
    print("Entrenamiento desactivado: no se cargan train ni val.")

## 5. Entrenar

Adam con `lr=0.001` y `weight_decay=0.0001` por defecto. Se aplica CrossEntropyLoss
sobre logits y se guarda el checkpoint de menor pérdida de validación.
Test no interviene en el entrenamiento ni en la selección del mejor modelo.
Cada ejecución de esta celda crea una carpeta nueva para sus resultados.

In [ ]:
def run_epoch(model, loader, device, optimizer=None, max_batches=None):
    training = optimizer is not None
    model.train(training)
    loss_sum = correct = total = 0
    criterion = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(training):
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                raise ValueError("La pérdida no es finita.")
            if training:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total

In [ ]:
checkpoint_path = Path(EXISTING_CHECKPOINT) if EXISTING_CHECKPOINT is not None else None
history = []
if RUN_TRAINING:
    # Una carpeta nueva evita mezclar resultados al repetir esta celda.
    training_dir = new_output(RUN_DIR / ("training_" + datetime.now().strftime("%H%M%S_%f")))
    checkpoint_path = training_dir / "best_model.pt"
    seed_everything(SEED)
    train_loader = make_loader(datasets["train"], EFFECTIVE_BATCH_SIZE, shuffle=True, seed=SEED)
    val_loader = make_loader(datasets["val"], EFFECTIVE_BATCH_SIZE)
    config = dict(data=str(DATA_DIR.resolve()), output=str(training_dir.resolve()),
                  epochs=TRAIN_EPOCHS, batch_size=EFFECTIVE_BATCH_SIZE, lr=LEARNING_RATE,
                  weight_decay=WEIGHT_DECAY, seed=SEED,
                  device_used=str(device), smoke_test=SMOKE_TEST, max_batches=MAX_BATCHES,
                  class_to_idx=datasets["train"].class_to_idx,
                  architecture="conv32-conv64-conv128-conv256-bn-gap-fc128-dropout0.5-fc2",
                  input_size=[3, 224, 224], torch_version=str(torch.__version__),
                  torchvision_version=str(__import__("torchvision").__version__))
    write_json(training_dir / "config.json", config)
    if SMOKE_TEST:
        print("PRUEBA TÉCNICA: las métricas no representan rendimiento final.")
    model = CatDogCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_loss = float("inf")
    with (training_dir / "history.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["epoch", "train_loss", "train_accuracy", "val_loss", "val_accuracy"])
        writer.writeheader()
        for epoch in range(1, TRAIN_EPOCHS + 1):
            train_loss, train_acc = run_epoch(model, train_loader, device, optimizer, MAX_BATCHES)
            val_loss, val_acc = run_epoch(model, val_loader, device, max_batches=MAX_BATCHES)
            row = dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_acc,
                       val_loss=val_loss, val_accuracy=val_acc)
            history.append(row)
            writer.writerow(row)
            handle.flush()
            print(f"Época {epoch}/{TRAIN_EPOCHS}: train loss={train_loss:.4f} acc={train_acc:.4f}; "
                  f"val loss={val_loss:.4f} acc={val_acc:.4f}", flush=True)
            if val_loss < best_loss:
                best_loss = val_loss
                torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
                            "val_loss": val_loss, "class_to_idx": datasets["train"].class_to_idx,
                            "config": config}, checkpoint_path)
    print("Mejor modelo:", checkpoint_path)
else:
    print("Entrenamiento omitido. Checkpoint:", checkpoint_path)

## 6. Curvas de aprendizaje

Train se mide durante el ajuste con Dropout activo; val se mide en modo evaluación.
Se muestran las curvas y se guardan en `curves.png` junto al checkpoint.

In [ ]:
if history:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ("loss", "accuracy")):
        for split in ("train", "val"):
            ax.plot([r["epoch"] for r in history], [r[f"{split}_{metric}"] for r in history], label=split)
        ax.set(xlabel="Época", ylabel=metric)
        ax.legend()
    fig.suptitle("PRUEBA TÉCNICA" if SMOKE_TEST else "Entrenamiento CNN")
    fig.tight_layout()
    fig.savefig(training_dir / "curves.png")
    plt.show()
    plt.close(fig)
else:
    print("No hay entrenamiento nuevo para graficar.")

## 7. Evaluar el mejor checkpoint

Se carga test directamente desde `DATA_DIR/test`. Se calculan accuracy, precision,
recall y F1 por clase y una matriz de confusión (filas reales, columnas predichas).
Usa test al finalizar la selección del modelo con validación.

En modo smoke se evalúan pocos lotes ordenados y pueden contener una sola clase;
sus resultados sirven únicamente para comprobar que el flujo funciona.

In [ ]:
if checkpoint_path is not None:
    evaluation_dir = new_output(RUN_DIR / ("evaluation_" + datetime.now().strftime("%H%M%S_%f")))
    evaluation_model, checkpoint = load_checkpoint(checkpoint_path, device)
    test = load_dataset(DATA_DIR, "test")
    truth, predictions = [], []
    with torch.inference_mode():
        for index, (images, labels) in enumerate(make_loader(test, EFFECTIVE_BATCH_SIZE)):
            if MAX_BATCHES is not None and index >= MAX_BATCHES:
                break
            predictions.extend(evaluation_model(images.to(device)).argmax(1).cpu().tolist())
            truth.extend(labels.tolist())
    smoke = checkpoint["config"].get("smoke_test", False) or SMOKE_TEST
    report = classification_report(truth, predictions, labels=[0, 1],
                                   target_names=test.classes, output_dict=True, zero_division=0)
    matrix = confusion_matrix(truth, predictions, labels=[0, 1])
    results = dict(smoke_test=smoke, checkpoint=str(checkpoint_path.resolve()),
                   checkpoint_epoch=checkpoint["epoch"], samples_evaluated=len(truth),
                   total_test_samples=len(test), accuracy=accuracy_score(truth, predictions),
                   classification_report=report, confusion_matrix=matrix.tolist())
    write_json(evaluation_dir / "metrics.json", results)
    if smoke:
        print("PRUEBA TÉCNICA: estas métricas no representan rendimiento final.")
    print(f"Accuracy: {results['accuracy']:.4f}; imágenes: {len(truth)}/{len(test)}")
    print(classification_report(truth, predictions, labels=[0, 1], target_names=test.classes, zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(matrix, display_labels=test.classes).plot(ax=ax, colorbar=False)
    ax.set_title("PRUEBA TÉCNICA" if smoke else "Evaluación en test")
    fig.tight_layout()
    fig.savefig(evaluation_dir / "confusion_matrix.png")
    plt.show()
    plt.close(fig)
    del evaluation_model
else:
    print("Evaluación omitida: entrena o configura EXISTING_CHECKPOINT.")

## 8. Archivos generados

Se guardan configuración, historial CSV, curvas, mejor checkpoint, métricas JSON y
matriz de confusión. Puedes descargar los archivos de salida al guardar una versión
del notebook en Kaggle. El dataset original no se copia ni se modifica.

In [ ]:
print("Directorio de esta ejecución:", RUN_DIR.resolve())
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(RUN_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
print("Pipeline completado.")

## 9. Probar el modelo terminado con una imagen nueva

Edita **`RUTA_IMAGEN`** en la siguiente celda y ejecútala tantas veces como quieras,
sin repetir el entrenamiento. Por defecto se utiliza `checkpoint_path`, el mejor
modelo guardado durante esta sesión. También puedes indicar **`RUTA_MODELO`** para
cargar un `best_model.pt` de una ejecución anterior o adjuntado a Kaggle.

La celda muestra la foto, la clase predicha y las probabilidades de gato y perro.
Opcionalmente, completa `CLASE_REAL` con `"cat"` o `"dog"` para comprobar el acierto.
Una imagen permite revisar un caso concreto; las métricas de test del bloque 7
miden el rendimiento sobre el conjunto completo. Las probabilidades softmax no
son una medida calibrada de certeza.

Si reiniciaste el kernel, ejecuta primero las celdas de bibliotecas, utilidades
compartidas y definición del modelo (bloques 1, 2 y 3), y especifica `RUTA_MODELO`.
No es necesario cargar el dataset ni entrenar de nuevo para usar un checkpoint existente.

In [ ]:
# Cambia esta ruta por la foto que quieres clasificar.
RUTA_IMAGEN = ""  # Ejemplo: "/kaggle/input/mis-fotos/mi_perro.jpg"
RUTA_MODELO = None  # None: mejor modelo de esta sesión; o "/kaggle/input/mi-modelo/best_model.pt"
CLASE_REAL = None  # Opcional: "cat" o "dog"

if not RUTA_IMAGEN:
    print("Escribe la ruta de tu foto en RUTA_IMAGEN y vuelve a ejecutar esta celda.")
else:
    ruta_imagen = Path(RUTA_IMAGEN).expanduser()
    if not ruta_imagen.is_file():
        raise FileNotFoundError(f"No existe la imagen: {ruta_imagen}")
    if CLASE_REAL is not None and CLASE_REAL not in {"cat", "dog"}:
        raise ValueError('CLASE_REAL debe ser "cat", "dog" o None')

    ruta_modelo = RUTA_MODELO if RUTA_MODELO is not None else globals().get("checkpoint_path")
    if ruta_modelo is None:
        raise ValueError("Indica RUTA_MODELO con la ruta del best_model.pt entrenado")
    ruta_modelo = Path(ruta_modelo).expanduser()
    if not ruta_modelo.is_file():
        raise FileNotFoundError(f"No existe el checkpoint: {ruta_modelo}")

    dispositivo_prueba = select_device("auto")
    modelo_prueba, checkpoint_prueba = load_checkpoint(ruta_modelo, dispositivo_prueba)
    with Image.open(ruta_imagen) as imagen:
        foto = ImageOps.exif_transpose(imagen).convert("RGB").copy()
        entrada = image_tensor(imagen).unsqueeze(0).to(dispositivo_prueba)
    with torch.inference_mode():
        probabilidades = modelo_prueba(entrada).softmax(dim=1)[0].cpu()

    clases_prueba = sorted(checkpoint_prueba["class_to_idx"],
                           key=checkpoint_prueba["class_to_idx"].get)
    clase_predicha = clases_prueba[probabilidades.argmax().item()]
    nombres_prueba = {"cat": "Gato", "dog": "Perro"}
    print("Modelo utilizado:", ruta_modelo)
    if checkpoint_prueba.get("config", {}).get("smoke_test", False):
        print("PRUEBA TÉCNICA: este checkpoint no tiene un entrenamiento completo.")
    print("Predicción:", nombres_prueba[clase_predicha])
    for clase, probabilidad in zip(clases_prueba, probabilidades.tolist()):
        print(f"{nombres_prueba[clase]}: {probabilidad:.2%}")
    if CLASE_REAL is not None:
        print("Clase real:", nombres_prueba[CLASE_REAL])
        print("Resultado:", "Acierto" if clase_predicha == CLASE_REAL else "Error")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(foto)
    ax.set_title(f"Predicción: {nombres_prueba[clase_predicha]}")
    ax.axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    del modelo_prueba